In [1]:
# Cell 1 — Imports and setup

import os
import json
from datetime import date
from pathlib import Path

# Verify artifacts directory exists
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

print("Artifacts directory:", artifacts_dir.resolve())
print("Today:", date.today())
print("Notebook: eda_12_evaluation_framework.ipynb")
print("Output target: artifacts/evaluation_checklist.md")
print()
print("Purpose: Write the evaluation checklist that model_14_signoff.ipynb")
print("will work through item by item. This notebook produces a document,")
print("not signal tests. No partial r. No feature verdicts.")
print()
print("Ready.")

Artifacts directory: /Users/kevinjohnson/cfb-analytics/artifacts
Today: 2026-05-09
Notebook: eda_12_evaluation_framework.ipynb
Output target: artifacts/evaluation_checklist.md

Purpose: Write the evaluation checklist that model_14_signoff.ipynb
will work through item by item. This notebook produces a document,
not signal tests. No partial r. No feature verdicts.

Ready.


In [2]:
# Cell 2 — Checklist definition
# Every item is a dict with: id, dimension, criterion, threshold, pass_condition, notes

checklist = {
    "metadata": {
        "project": "CFB Analytics — Hierarchical Negative Binomial Model",
        "notebook": "eda_12_evaluation_framework.ipynb",
        "created": str(date.today()),
        "purpose": (
            "Defines pass/fail criteria for model sign-off. "
            "Day 33 (model_14_signoff.ipynb) works through every item. "
            "Model is not signed off until every item has an explicit PASS "
            "or a documented exception with justification."
        ),
        "go_live": "2026-09-24",
    },

    "dimensions": [

        # ── 1. CONVERGENCE AND SAMPLING QUALITY ──────────────────────────────
        {
            "id": "CONV-01",
            "dimension": "Convergence and Sampling Quality",
            "criterion": "R-hat for all parameters",
            "threshold": "R-hat < 1.01 for every parameter in the model",
            "pass_condition": "max(R-hat) < 1.01 with no individual parameter exceeding 1.01",
            "notes": (
                "Check team-level attack and defense parameters individually — "
                "do not rely on summary statistics alone. If any parameter exceeds "
                "1.01, investigate before proceeding. R-hat between 1.01 and 1.05 "
                "requires documented justification. R-hat > 1.05 is an automatic FAIL."
            ),
        },
        {
            "id": "CONV-02",
            "dimension": "Convergence and Sampling Quality",
            "criterion": "Effective sample size",
            "threshold": "ESS_bulk >= 400 and ESS_tail >= 400 for all parameters",
            "pass_condition": "min(ESS_bulk) >= 400 and min(ESS_tail) >= 400",
            "notes": (
                "ESS between 200 and 400 requires documented justification. "
                "ESS < 200 for any parameter is an automatic FAIL. "
                "Pay particular attention to dispersion parameter r and "
                "conference-level hyperparameters."
            ),
        },
        {
            "id": "CONV-03",
            "dimension": "Convergence and Sampling Quality",
            "criterion": "Divergences",
            "threshold": "Divergence count = 0 after tuning",
            "pass_condition": "Zero divergences in the post-warmup sample",
            "notes": (
                "Any divergences require investigation of the model geometry — "
                "do not simply increase target_accept and rerun. "
                "If divergences persist after reparameterization attempts, "
                "document the structural cause before proceeding."
            ),
        },
        {
            "id": "CONV-04",
            "dimension": "Convergence and Sampling Quality",
            "criterion": "Trace plots",
            "threshold": "All chains mix visually — no trends, no chain separation, no sticking",
            "pass_condition": (
                "Visual inspection of trace plots for all top-level parameters "
                "shows stationary, well-mixed chains"
            ),
            "notes": (
                "Inspect at minimum: league-level intercept, conference-level "
                "hyperparameters, dispersion parameter r, home field advantage, "
                "and at least one team-level attack and defense parameter per conference. "
                "Document any chain that shows non-stationarity."
            ),
        },
        {
            "id": "CONV-05",
            "dimension": "Convergence and Sampling Quality",
            "criterion": "Energy plots (BFMI)",
            "threshold": "BFMI > 0.3 for all chains",
            "pass_condition": "min(BFMI across chains) > 0.3",
            "notes": (
                "BFMI between 0.2 and 0.3 requires documented justification. "
                "BFMI < 0.2 indicates the sampler is not exploring the posterior "
                "efficiently and is an automatic FAIL."
            ),
        },

        # ── 2. PRIOR PREDICTIVE CHECKS ────────────────────────────────────────
        {
            "id": "PRIOR-01",
            "dimension": "Prior Predictive Checks",
            "criterion": "Score range plausibility",
            "threshold": (
                "95% of prior predictive samples fall within 0–70 points per team. "
                "Zero samples below 0. Fewer than 1% of samples above 80 points."
            ),
            "pass_condition": (
                "Prior predictive distribution produces no negative scores, "
                "fewer than 1% of samples above 80, and 95th percentile <= 70"
            ),
            "notes": (
                "Sample at least 500 prior predictive games. "
                "If the prior generates 0-point or 150-point games, "
                "the priors are miscalibrated — fix before fitting. "
                "This check runs before any data is seen."
            ),
        },
        {
            "id": "PRIOR-02",
            "dimension": "Prior Predictive Checks",
            "criterion": "Score variance consistency with observed VMR",
            "threshold": "Prior predictive VMR falls within 3.0–10.0",
            "pass_condition": (
                "Variance-to-mean ratio of prior predictive score samples "
                "is between 3.0 and 10.0"
            ),
            "notes": (
                "Observed VMR range from EDA: 4.95–7.16 (2022–2024). "
                "Prior predictive VMR should be wider than observed (priors are uncertain) "
                "but not implausibly wide. VMR < 2.0 suggests under-dispersed priors. "
                "VMR > 15.0 suggests over-dispersed priors."
            ),
        },
        {
            "id": "PRIOR-03",
            "dimension": "Prior Predictive Checks",
            "criterion": "Plausible total points distribution",
            "threshold": (
                "Prior predictive mean total points between 40 and 65. "
                "95% interval does not extend below 10 or above 120."
            ),
            "pass_condition": (
                "40 <= mean(prior predictive total points) <= 65 "
                "and P2.5 >= 10 and P97.5 <= 120"
            ),
            "notes": (
                "Observed mean total points 2022–2024: approximately 52–56. "
                "Prior mean should be in plausible CFB territory. "
                "Extremes indicate a prior specification error."
            ),
        },

        # ── 3. POSTERIOR PREDICTIVE CHECKS ───────────────────────────────────
        {
            "id": "POST-01",
            "dimension": "Posterior Predictive Checks",
            "criterion": "Overall score distribution fit",
            "threshold": (
                "Posterior predictive mean within ±2 points of observed mean. "
                "Posterior predictive SD within ±3 points of observed SD."
            ),
            "pass_condition": (
                "abs(posterior_predictive_mean - observed_mean) <= 2.0 "
                "and abs(posterior_predictive_sd - observed_sd) <= 3.0"
            ),
            "notes": (
                "Compute separately for points_scored and points_allowed. "
                "A model that fits the mean but not the variance has a dispersion problem."
            ),
        },
        {
            "id": "POST-02",
            "dimension": "Posterior Predictive Checks",
            "criterion": "Conference-level calibration",
            "threshold": (
                "Posterior predictive mean within ±3 points of observed conference mean "
                "for all 10 FBS conferences individually"
            ),
            "pass_condition": (
                "All 10 conferences pass the ±3 point mean check. "
                "Flag any conference outside ±5 points as a hard FAIL."
            ),
            "notes": (
                "Check all 10 conferences: ACC, American Athletic, Big 12, Big Ten, "
                "Conference USA, Mid-American, Mountain West, Pac-12, SEC, Sun Belt. "
                "A conference that consistently over- or under-predicts has a "
                "conference-level prior or pooling problem."
            ),
        },
        {
            "id": "POST-03",
            "dimension": "Posterior Predictive Checks",
            "criterion": "Dispersion parameter adequacy",
            "threshold": (
                "Posterior predictive VMR per conference within 1.5x of observed VMR "
                "for that conference"
            ),
            "pass_condition": (
                "max(posterior_predictive_VMR / observed_VMR) <= 1.5 "
                "and min(posterior_predictive_VMR / observed_VMR) >= 0.67 "
                "across all conferences"
            ),
            "notes": (
                "If any conference shows systematic VMR miscalibration, "
                "add conference-specific dispersion parameter r as specified in "
                "the model architecture decision (Day 10). "
                "Single r is the starting assumption — this check is the gate "
                "for whether that assumption holds."
            ),
        },
        {
            "id": "POST-04",
            "dimension": "Posterior Predictive Checks",
            "criterion": "Tail behavior — high-scoring and low-scoring games",
            "threshold": (
                "Posterior predictive frequency of games with total points >= 70 "
                "within ±3pp of observed frequency. "
                "Posterior predictive frequency of games with total points <= 30 "
                "within ±3pp of observed frequency."
            ),
            "pass_condition": (
                "Both tail frequency checks pass within ±3 percentage points"
            ),
            "notes": (
                "Tail calibration matters for over/under prediction. "
                "A model that gets the mean right but underestimates tail probability "
                "will systematically miscalibrate over/under lines in extreme games."
            ),
        },

        # ── 4. HOLDOUT EVALUATION ─────────────────────────────────────────────
        {
            "id": "HOLD-01",
            "dimension": "Holdout Evaluation (2025 Season)",
            "criterion": "Overall Brier score vs baseline",
            "threshold": "Brier score lower than a naive baseline using only SP+ delta",
            "pass_condition": (
                "model_brier_score < sp_only_baseline_brier_score "
                "on the full 2025 holdout set"
            ),
            "notes": (
                "The SP+-only baseline uses pregame SP+ rating differential to generate "
                "win probabilities via logistic regression. If the full model does not "
                "beat this baseline, the additional features are not earning their complexity. "
                "Also report vs a coin-flip baseline (Brier = 0.25) for reference."
            ),
        },
        {
            "id": "HOLD-02",
            "dimension": "Holdout Evaluation (2025 Season)",
            "criterion": "Calibration curve — win probability",
            "threshold": (
                "Predicted win probabilities within ±5pp of observed win frequencies "
                "across all decile buckets (0–10%, 10–20%, ... 90–100%)"
            ),
            "pass_condition": (
                "All decile buckets with n >= 20 games pass the ±5pp check. "
                "No bucket with n >= 20 exceeds ±8pp."
            ),
            "notes": (
                "Buckets with n < 20 are reported but not gated. "
                "Systematic over-confidence (predicted > observed in high-probability buckets) "
                "and systematic under-confidence are distinct failure modes — report separately."
            ),
        },
        {
            "id": "HOLD-03",
            "dimension": "Holdout Evaluation (2025 Season)",
            "criterion": "Spread accuracy by expected margin bucket",
            "threshold": (
                "Mean absolute spread error <= 14 points overall. "
                "No expected-margin bucket (0–7, 7–14, 14–21, 21+) "
                "has MAE exceeding 18 points."
            ),
            "pass_condition": (
                "overall_MAE <= 14.0 "
                "and max(bucket_MAE across margin buckets with n >= 20) <= 18.0"
            ),
            "notes": (
                "MAE thresholds calibrated against typical CFB spread accuracy benchmarks. "
                "If the model consistently underestimates margin in blowouts (21+ bucket), "
                "the dispersion parameter or team-strength hierarchy has a structural problem."
            ),
        },
        {
            "id": "HOLD-04",
            "dimension": "Holdout Evaluation (2025 Season)",
            "criterion": "Over/under calibration",
            "threshold": (
                "Predicted over probability within ±5pp of observed over frequency "
                "across total-points decile buckets"
            ),
            "pass_condition": (
                "All decile buckets with n >= 20 games pass the ±5pp check "
                "on over/under calibration"
            ),
            "notes": (
                "Report separately for dome games vs outdoor games and for "
                "games with wind_chill <= 25°F — these are the subpopulations "
                "where environmental features are expected to contribute."
            ),
        },

        # ── 5. SUBGROUP EVALUATION ────────────────────────────────────────────
        {
            "id": "SUB-01",
            "dimension": "Subgroup Evaluation",
            "criterion": "Brier score by conference tier — P4",
            "threshold": "P4 Brier score <= 0.23",
            "pass_condition": "p4_brier_score <= 0.23",
            "notes": (
                "P4 conferences: ACC, Big 12, Big Ten, SEC. "
                "P4 games have more data per team, so the model should calibrate "
                "better here than in G5. Threshold is tighter than overall."
            ),
        },
        {
            "id": "SUB-02",
            "dimension": "Subgroup Evaluation",
            "criterion": "Brier score by conference tier — G5",
            "threshold": "G5 Brier score <= 0.25",
            "pass_condition": "g5_brier_score <= 0.25",
            "notes": (
                "G5 conferences: American Athletic, Conference USA, Mid-American, "
                "Mountain West, Pac-12 (remnant), Sun Belt. "
                "G5 teams have thinner data — some degradation vs P4 is acceptable, "
                "but the model must remain credible. "
                "If G5 Brier > 0.25, investigate whether conference-level pooling "
                "is providing adequate regularization."
            ),
        },
        {
            "id": "SUB-03",
            "dimension": "Subgroup Evaluation",
            "criterion": "Calibration by individual conference",
            "threshold": (
                "Win probability calibration within ±8pp of observed frequency "
                "for each of the 10 FBS conferences individually "
                "(in decile buckets with n >= 10)"
            ),
            "pass_condition": (
                "All 10 conferences pass the ±8pp calibration check "
                "in buckets with n >= 10"
            ),
            "notes": (
                "Report all 10 conferences: ACC, American Athletic, Big 12, Big Ten, "
                "Conference USA, Mid-American, Mountain West, Pac-12, SEC, Sun Belt. "
                "A conference that systematically fails calibration despite passing overall "
                "indicates a conference-level prior or pooling problem."
            ),
        },
        {
            "id": "SUB-04",
            "dimension": "Subgroup Evaluation",
            "criterion": "Rivalry games",
            "threshold": "Brier score for rivalry games within ±0.03 of overall Brier score",
            "pass_condition": (
                "abs(rivalry_brier_score - overall_brier_score) <= 0.03"
            ),
            "notes": (
                "Rivalry games are defined from the raw.games rivalry flag or a "
                "curated list. These games have higher upset rates and crowd effects. "
                "If rivalry_brier >> overall_brier, the model is not accounting for "
                "rivalry dynamics. Report rivalry upset rate vs model-implied upset rate."
            ),
        },
        {
            "id": "SUB-05",
            "dimension": "Subgroup Evaluation",
            "criterion": "Cross-tier matchups (P4 vs G5)",
            "threshold": (
                "Brier score for cross-tier games <= 0.22. "
                "Mean predicted win probability for P4 favorites between 0.72 and 0.88."
            ),
            "pass_condition": (
                "cross_tier_brier_score <= 0.22 "
                "and 0.72 <= mean_p4_win_prob_vs_g5 <= 0.88"
            ),
            "notes": (
                "Cross-tier matchups are the games where the model's strength separation "
                "is most directly tested. If the model assigns P4 teams < 72% win probability "
                "on average vs G5, it is under-differentiating. "
                "If > 88%, it is over-differentiating. "
                "Both are calibration failures."
            ),
        },
        {
            "id": "SUB-06",
            "dimension": "Subgroup Evaluation",
            "criterion": "Neutral site games",
            "threshold": (
                "Brier score for neutral site games within ±0.03 of overall Brier score. "
                "Mean predicted home advantage for neutral site games between -1.5 and +1.5 points."
            ),
            "pass_condition": (
                "abs(neutral_site_brier_score - overall_brier_score) <= 0.03 "
                "and abs(mean_neutral_site_home_advantage) <= 1.5"
            ),
            "notes": (
                "Neutral site games have no home field advantage. "
                "If the model systematically assigns home advantage to neutral games, "
                "the home field feature is not correctly conditioned on venue type."
            ),
        },
        {
            "id": "SUB-07",
            "dimension": "Subgroup Evaluation",
            "criterion": "Season progression — conference game 1 (prior-driven)",
            "threshold": (
                "Brier score for conf game 1 <= 0.26. "
                "Calibration within ±10pp of observed frequency."
            ),
            "pass_condition": (
                "conf_game_1_brier_score <= 0.26 "
                "and conf_game_1_calibration_error <= 0.10"
            ),
            "notes": (
                "At conf game 1, rolling features are null — model runs entirely "
                "on prior (SP+, recruiting) and pregame ELO. "
                "Degraded performance vs later games is expected. "
                "The gate here is that the model produces credible estimates, "
                "not that it matches later-season accuracy."
            ),
        },
        {
            "id": "SUB-08",
            "dimension": "Subgroup Evaluation",
            "criterion": "Season progression — conference games 5–8 (posterior-informed)",
            "threshold": "Brier score for conf games 5–8 <= 0.23",
            "pass_condition": "conf_games_5_8_brier_score <= 0.23",
            "notes": (
                "By games 5–8 the posterior has accumulated meaningful in-season data. "
                "If Brier does not improve from conf game 1 to games 5–8, "
                "the rolling features are not contributing. "
                "Report improvement trajectory: game 1, games 2–4, games 5–8, games 9–12."
            ),
        },

        # ── 6. EDGE CASE STRESS TESTS ─────────────────────────────────────────
        {
            "id": "EDGE-01",
            "dimension": "Edge Case Stress Tests",
            "criterion": "Extreme elevation differential (>= 2000ft threshold)",
            "threshold": (
                "Spread MAE for elevation-triggered games (away_elevation_delta_ft >= 2000) "
                "within ±4 points of overall spread MAE"
            ),
            "pass_condition": (
                "abs(elevation_triggered_MAE - overall_MAE) <= 4.0"
            ),
            "notes": (
                "EDA confirmed signal concentrates in Mountain West and Big 12 "
                "above the 2000ft threshold. YoY r = 0.8255. "
                "If elevation-triggered games show systematically higher MAE, "
                "the threshold-activated feature is not correctly specified."
            ),
        },
        {
            "id": "EDGE-02",
            "dimension": "Edge Case Stress Tests",
            "criterion": "Extreme travel distance (>= 1500mi threshold)",
            "threshold": (
                "Spread MAE for travel-triggered games (away_travel_distance_mi >= 1500) "
                "within ±4 points of overall spread MAE"
            ),
            "pass_condition": (
                "abs(travel_triggered_MAE - overall_MAE) <= 4.0"
            ),
            "notes": (
                "EDA confirmed spread signal only — no O/U signal. "
                "YoY r = 0.6562 (below anchor threshold but included as supporting). "
                "Report n for this subpopulation — may be small in 2025 holdout."
            ),
        },
        {
            "id": "EDGE-03",
            "dimension": "Edge Case Stress Tests",
            "criterion": "Large timezone delta (abs >= 2hr threshold)",
            "threshold": (
                "Spread MAE for timezone-triggered games (abs(away_tz_delta_hrs) >= 2) "
                "within ±4 points of overall spread MAE"
            ),
            "pass_condition": (
                "abs(timezone_triggered_MAE - overall_MAE) <= 4.0"
            ),
            "notes": (
                "EDA confirmed spread signal only. "
                "Signal strengthens at abs >= 3hr (r = -0.3103, n = 38). "
                "Report separately for abs >= 2hr and abs >= 3hr subpopulations."
            ),
        },
        {
            "id": "EDGE-04",
            "dimension": "Edge Case Stress Tests",
            "criterion": "Extreme cold weather (wind_chill <= 25°F)",
            "threshold": (
                "Over/under calibration error for cold-weather games (wind_chill <= 25°F) "
                "within ±8pp of observed over frequency"
            ),
            "pass_condition": (
                "cold_weather_ou_calibration_error <= 0.08"
            ),
            "notes": (
                "EDA confirmed O/U signal only at wind_chill <= 25°F (r = 0.3373, n = 71). "
                "No spread signal. "
                "If the model does not lower total point predictions in extreme cold, "
                "the wind_chill feature is not activating correctly."
            ),
        },
        {
            "id": "EDGE-05",
            "dimension": "Edge Case Stress Tests",
            "criterion": "Teams with thin conference game history in training data",
            "threshold": (
                "Brier score for games involving teams with <= 10 conference games "
                "in 2022–2024 training data within ±0.04 of overall Brier score"
            ),
            "pass_condition": (
                "abs(thin_data_team_brier_score - overall_brier_score) <= 0.04"
            ),
            "notes": (
                "Thin-data teams include: teams that joined a conference late in the "
                "training window, teams that played reduced schedules, and new FBS programs. "
                "Conference-level pooling should regularize these teams toward the "
                "conference mean. If thin-data games show substantially higher Brier, "
                "pooling is insufficient."
            ),
        },
        {
            "id": "EDGE-06",
            "dimension": "Edge Case Stress Tests",
            "criterion": "Teams that changed conferences between seasons",
            "threshold": (
                "Brier score for games involving conference-switching teams "
                "within ±0.04 of overall Brier score"
            ),
            "pass_condition": (
                "abs(conference_switcher_brier_score - overall_brier_score) <= 0.04"
            ),
            "notes": (
                "Conference assignment is historically accurate by season (locked decision). "
                "Teams switching conferences between 2024 and 2025 appear in a new "
                "conference pool in 2025 with only prior-based estimates. "
                "The model must handle this gracefully via SP+ and EPA priors."
            ),
        },

        # ── 7. FEATURE CONTRIBUTION CHECKS ───────────────────────────────────
        {
            "id": "FEAT-01",
            "dimension": "Feature Contribution Checks",
            "criterion": "Close-game EPA anchor pair — direction",
            "threshold": (
                "Posterior mean coefficient for close_game_epa_per_play is positive. "
                "Posterior mean coefficient for close_game_def_epa_per_play is negative. "
                "Both 94% HDI exclude zero."
            ),
            "pass_condition": (
                "close_game_off_epa_coef > 0 with 94% HDI excluding zero "
                "and close_game_def_epa_coef < 0 with 94% HDI excluding zero"
            ),
            "notes": (
                "EDA anchor: off EPA r = +0.5988 with point_differential, "
                "def EPA r = -0.6134. "
                "If either coefficient is in the wrong direction, the model has "
                "a specification error. If either HDI includes zero, the feature "
                "is not contributing — investigate collinearity."
            ),
        },
        {
            "id": "FEAT-02",
            "dimension": "Feature Contribution Checks",
            "criterion": "SP+ prior weight — present at game 1, not aggressively decayed",
            "threshold": (
                "Effective SP+ weight at conf game 1 >= 0.15 (partial r equivalent). "
                "Effective SP+ weight at conf games 9–12 >= 0.10. "
                "Weight does not drop below 0.10 at any point in the season arc."
            ),
            "pass_condition": (
                "sp_weight_game_1 >= 0.15 "
                "and sp_weight_games_9_12 >= 0.10 "
                "and min(sp_weight_season_arc) >= 0.10"
            ),
            "notes": (
                "EDA finding (Day 9): SP+ partial r = 0.2240 at conf game 1 "
                "after EPA control. Games 9–12 r = 0.2609 — prior remains relevant "
                "throughout season. Do not aggressively down-weight SP+ as games accumulate. "
                "Measure effective weight via posterior sensitivity analysis or "
                "leave-one-out comparison."
            ),
        },
        {
            "id": "FEAT-03",
            "dimension": "Feature Contribution Checks",
            "criterion": "Conference-specific features fire only in confirmed conferences",
            "threshold": (
                "Posterior mean coefficient for each conference-specific feature "
                "is meaningfully non-zero (94% HDI excludes zero) only in the "
                "conferences where EDA confirmed signal"
            ),
            "pass_condition": (
                "Conference-specific features do not show strong posterior weight "
                "in conferences where EDA found null signal"
            ),
            "notes": (
                "Examples: last3_off_epa_avg confirmed in ACC, Conference USA, "
                "Mid-American, Mountain West, SEC — not in American Athletic, "
                "Big 12, Big Ten, Pac-12, Sun Belt. "
                "days_since_last_game confirmed in American Athletic and Big 12 only. "
                "If a feature fires broadly in non-confirmed conferences, "
                "the conference-level pooling structure is not correctly specified."
            ),
        },
        {
            "id": "FEAT-04",
            "dimension": "Feature Contribution Checks",
            "criterion": "Threshold-activated features activate correctly",
            "threshold": (
                "Elevation, travel, and timezone features show near-zero posterior "
                "contribution in games below their respective thresholds "
                "and meaningfully non-zero contribution above thresholds"
            ),
            "pass_condition": (
                "Below-threshold posterior contribution indistinguishable from zero. "
                "Above-threshold posterior contribution 94% HDI excludes zero."
            ),
            "notes": (
                "Thresholds: elevation >= 2000ft, travel >= 1500mi, "
                "timezone abs >= 2hr, wind_chill <= 25°F. "
                "These are modeled as indicator×magnitude interactions. "
                "Verify the indicator is correctly coded — a linear specification "
                "would spread signal across the full range and dilute the threshold effect."
            ),
        },
        {
            "id": "FEAT-05",
            "dimension": "Feature Contribution Checks",
            "criterion": "Recruiting prior — Sun Belt direction check",
            "threshold": (
                "Posterior mean recruiting prior weight for Sun Belt is <= 0 "
                "or recruiting is excluded from the Sun Belt prior entirely"
            ),
            "pass_condition": (
                "Sun Belt recruiting prior weight is non-positive "
                "or feature is correctly excluded for Sun Belt"
            ),
            "notes": (
                "EDA finding (Day 9): Sun Belt rec↔diff_r = -0.2665. "
                "Recruiting is negatively correlated with outcomes in Sun Belt. "
                "Using it as a positive prior signal in Sun Belt is a specification error. "
                "Either exclude recruiting from the Sun Belt prior or allow the "
                "conference-specific weight to be negative."
            ),
        },

        # ── 8. KNOWN FAILURE MODES ────────────────────────────────────────────
        {
            "id": "FAIL-01",
            "dimension": "Known Failure Modes",
            "criterion": "Conf game 1 without rolling features — graceful handling",
            "threshold": (
                "No null predictions at conf game 1. "
                "Model produces a complete score distribution for every game "
                "in the holdout set regardless of rolling feature availability."
            ),
            "pass_condition": (
                "Zero null or degenerate predictions (point mass, infinite variance) "
                "at conf game 1 in the 2025 holdout"
            ),
            "notes": (
                "Rolling features are null at conf game 1. "
                "Null handling strategy: Approach A — impute with season-to-date prior "
                "(locked decision). Verify imputation is correctly applied and that "
                "the model does not silently propagate nulls into the likelihood."
            ),
        },
        {
            "id": "FAIL-02",
            "dimension": "Known Failure Modes",
            "criterion": "G5 teams with thin data — reasonable estimates",
            "threshold": (
                "No G5 team with <= 10 training games produces a posterior mean "
                "score more than 20 points from its conference mean"
            ),
            "pass_condition": (
                "max(abs(thin_g5_team_posterior_mean - conference_mean)) <= 20.0"
            ),
            "notes": (
                "Conference-level pooling should pull thin-data teams toward the "
                "conference mean. A team with 3–5 training games should not receive "
                "extreme posterior estimates. If it does, the pooling hyperprior "
                "is too weak or the team-level variance is too high."
            ),
        },
        {
            "id": "FAIL-03",
            "dimension": "Known Failure Modes",
            "criterion": "Cross-tier matchups — no extreme miscalibration",
            "threshold": (
                "No cross-tier game produces a posterior win probability outside "
                "the range 0.05–0.98 for the stronger team. "
                "Mean absolute calibration error for cross-tier games <= 0.10."
            ),
            "pass_condition": (
                "All cross-tier posterior win probabilities within 0.05–0.98 "
                "and cross_tier_mean_calibration_error <= 0.10"
            ),
            "notes": (
                "Win probabilities of 0.99+ or 0.01- are computationally generated "
                "certainty that CFB does not support. Even the most extreme mismatches "
                "carry upset probability. Hard-code a floor/ceiling if necessary, "
                "but investigate the root cause — this usually indicates the team-level "
                "attack/defense parameters are diverging."
            ),
        },
        {
            "id": "FAIL-04",
            "dimension": "Known Failure Modes",
            "criterion": "Sun Belt recruiting prior direction",
            "threshold": "See FEAT-05",
            "pass_condition": "See FEAT-05",
            "notes": (
                "Duplicate gate: Sun Belt recruiting prior direction is critical enough "
                "to check from both the feature contribution angle (FEAT-05) and the "
                "known failure mode angle (FAIL-04). Day 33 must check both items "
                "and document that both were reviewed."
            ),
        },
    ]
}

# Summary count
dims = {}
for item in checklist["dimensions"]:
    d = item["dimension"]
    dims[d] = dims.get(d, 0) + 1

print(f"Total checklist items: {len(checklist['dimensions'])}")
print()
print("Items by dimension:")
for d, count in dims.items():
    print(f"  {count:2d}  {d}")

Total checklist items: 39

Items by dimension:
   5  Convergence and Sampling Quality
   3  Prior Predictive Checks
   4  Posterior Predictive Checks
   4  Holdout Evaluation (2025 Season)
   8  Subgroup Evaluation
   6  Edge Case Stress Tests
   5  Feature Contribution Checks
   4  Known Failure Modes


In [3]:
# Cell 3 — Render checklist to markdown and write to artifacts/evaluation_checklist.md

def render_checklist(checklist: dict) -> str:
    meta = checklist["metadata"]
    items = checklist["dimensions"]

    lines = []

    # ── Header ────────────────────────────────────────────────────────────────
    lines.append(f"# CFB Analytics — Model Evaluation Checklist")
    lines.append(f"")
    lines.append(f"**Project:** {meta['project']}  ")
    lines.append(f"**Created:** {meta['created']}  ")
    lines.append(f"**Source notebook:** `eda_12_evaluation_framework.ipynb`  ")
    lines.append(f"**Go-live date:** {meta['go_live']}  ")
    lines.append(f"")
    lines.append(f"## Purpose")
    lines.append(f"")
    lines.append(meta["purpose"])
    lines.append(f"")
    lines.append(
        f"**Total items:** {len(items)}  \n"
        f"**Sign-off notebook:** `model_14_signoff.ipynb` (Day 33)  \n"
        f"**Sign-off rule:** Every item must record PASS or a documented exception "
        f"with explicit justification. No item may be skipped."
    )
    lines.append(f"")

    # ── Table of contents ─────────────────────────────────────────────────────
    lines.append(f"## Dimensions")
    lines.append(f"")

    dims_seen = []
    dim_counts = {}
    for item in items:
        d = item["dimension"]
        if d not in dims_seen:
            dims_seen.append(d)
        dim_counts[d] = dim_counts.get(d, 0) + 1

    for d in dims_seen:
        anchor = d.lower().replace(" ", "-").replace("(", "").replace(")", "").replace("/", "").replace("–", "")
        lines.append(f"- [{d} ({dim_counts[d]} items)](#{anchor})")

    lines.append(f"")
    lines.append(f"---")
    lines.append(f"")

    # ── Checklist items grouped by dimension ──────────────────────────────────
    current_dim = None
    for item in items:
        if item["dimension"] != current_dim:
            current_dim = item["dimension"]
            lines.append(f"## {current_dim}")
            lines.append(f"")

        lines.append(f"### {item['id']} — {item['criterion']}")
        lines.append(f"")
        lines.append(f"**Threshold:** {item['threshold']}")
        lines.append(f"")
        lines.append(f"**Pass condition:**")
        lines.append(f"")
        lines.append(f"```")
        lines.append(item["pass_condition"])
        lines.append(f"```")
        lines.append(f"")
        lines.append(f"**Notes:** {item['notes']}")
        lines.append(f"")
        lines.append(f"**Day 33 result:** ☐ PASS &nbsp;&nbsp; ☐ FAIL &nbsp;&nbsp; ☐ EXCEPTION")
        lines.append(f"")
        lines.append(f"**Day 33 notes:**")
        lines.append(f"")
        lines.append(f"> *(record finding here)*")
        lines.append(f"")
        lines.append(f"---")
        lines.append(f"")

    # ── Sign-off block ────────────────────────────────────────────────────────
    lines.append(f"## Sign-Off")
    lines.append(f"")
    lines.append(
        f"All 39 items must be marked PASS or EXCEPTION before sign-off is granted. "
        f"An EXCEPTION requires an explicit written justification explaining why the "
        f"failure does not disqualify the model from production use. "
        f"Undocumented failures are not exceptions — they are failures."
    )
    lines.append(f"")
    lines.append(f"| Field | Value |")
    lines.append(f"|---|---|")
    lines.append(f"| Items passed | |")
    lines.append(f"| Items failed | |")
    lines.append(f"| Documented exceptions | |")
    lines.append(f"| Sign-off granted | ☐ YES &nbsp; ☐ NO |")
    lines.append(f"| Signed off by | |")
    lines.append(f"| Sign-off date | |")
    lines.append(f"| Notebook | `model_14_signoff.ipynb` |")
    lines.append(f"")

    return "\n".join(lines)


# Render and write
md_content = render_checklist(checklist)
output_path = artifacts_dir / "evaluation_checklist.md"

with open(output_path, "w") as f:
    f.write(md_content)

# Verify
written_size = output_path.stat().st_size
line_count = md_content.count("\n")

print(f"Written: {output_path}")
print(f"Size: {written_size:,} bytes")
print(f"Lines: {line_count:,}")
print()

# Spot-check: confirm all 39 IDs are present in the file
missing = []
for item in checklist["dimensions"]:
    if item["id"] not in md_content:
        missing.append(item["id"])

if missing:
    print(f"MISSING IDs: {missing}")
else:
    print(f"All 39 item IDs confirmed present in output file.")

Written: artifacts/evaluation_checklist.md
Size: 27,712 bytes
Lines: 837

All 39 item IDs confirmed present in output file.


In [4]:
# Cell 4 — Structural audit of artifacts/evaluation_checklist.md

import re

with open(artifacts_dir / "evaluation_checklist.md", "r") as f:
    md = f.read()

errors = []
warnings = []

# ── 1. Required top-level sections ───────────────────────────────────────────
required_sections = [
    "## Purpose",
    "## Dimensions",
    "## Sign-Off",
]
for section in required_sections:
    if section not in md:
        errors.append(f"Missing required section: '{section}'")

# ── 2. Every item ID present with required fields ─────────────────────────────
required_fields = [
    "**Threshold:**",
    "**Pass condition:**",
    "**Notes:**",
    "**Day 33 result:**",
    "**Day 33 notes:**",
]

for item in checklist["dimensions"]:
    item_id = item["id"]

    # Find the block for this item
    pattern = rf"### {re.escape(item_id)} —.*?(?=### [A-Z]{{2,}}-\d{{2}}|## Sign-Off)"
    match = re.search(pattern, md, re.DOTALL)

    if not match:
        errors.append(f"{item_id}: block not found in rendered markdown")
        continue

    block = match.group(0)

    for field in required_fields:
        if field not in block:
            errors.append(f"{item_id}: missing field '{field}'")

    # Day 33 result line must have all three checkboxes
    if "☐ PASS" not in block:
        errors.append(f"{item_id}: missing PASS checkbox")
    if "☐ FAIL" not in block:
        errors.append(f"{item_id}: missing FAIL checkbox")
    if "☐ EXCEPTION" not in block:
        errors.append(f"{item_id}: missing EXCEPTION checkbox")

    # Pass condition must be in a code block
    code_block_pattern = r"```\n.*?\n```"
    if not re.search(code_block_pattern, block, re.DOTALL):
        errors.append(f"{item_id}: pass condition not in code block")

# ── 3. All 8 dimensions present as ## headers ─────────────────────────────────
expected_dimensions = [
    "Convergence and Sampling Quality",
    "Prior Predictive Checks",
    "Posterior Predictive Checks",
    "Holdout Evaluation (2025 Season)",
    "Subgroup Evaluation",
    "Edge Case Stress Tests",
    "Feature Contribution Checks",
    "Known Failure Modes",
]
for dim in expected_dimensions:
    if f"## {dim}" not in md:
        errors.append(f"Missing dimension header: '## {dim}'")

# ── 4. Sign-off block has required table fields ───────────────────────────────
signoff_fields = [
    "Items passed",
    "Items failed",
    "Documented exceptions",
    "Sign-off granted",
    "Signed off by",
    "Sign-off date",
    "model_14_signoff.ipynb",
]
for field in signoff_fields:
    if field not in md:
        errors.append(f"Sign-off block missing field: '{field}'")

# ── 5. ID sequence integrity ──────────────────────────────────────────────────
expected_ids = [item["id"] for item in checklist["dimensions"]]
found_ids = re.findall(r"### ([A-Z]+-\d+) —", md)

if found_ids != expected_ids:
    # Find specific mismatches
    if len(found_ids) != len(expected_ids):
        errors.append(
            f"ID count mismatch: expected {len(expected_ids)}, found {len(found_ids)}"
        )
    else:
        for i, (exp, got) in enumerate(zip(expected_ids, found_ids)):
            if exp != got:
                errors.append(f"ID sequence error at position {i}: expected {exp}, got {got}")

# ── 6. No placeholder text left unfilled ─────────────────────────────────────
placeholders = ["TODO", "FIXME", "PLACEHOLDER", "TBD"]
for p in placeholders:
    if p in md:
        warnings.append(f"Placeholder text found: '{p}'")

# ── Report ────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STRUCTURAL AUDIT — artifacts/evaluation_checklist.md")
print("=" * 60)
print()

if errors:
    print(f"ERRORS ({len(errors)}):")
    for e in errors:
        print(f"  ✗ {e}")
else:
    print("ERRORS: none")

print()

if warnings:
    print(f"WARNINGS ({len(warnings)}):")
    for w in warnings:
        print(f"  ⚠ {w}")
else:
    print("WARNINGS: none")

print()
print(f"Items audited:     {len(checklist['dimensions'])}")
print(f"Dimensions checked: {len(expected_dimensions)}")
print(f"Sign-off fields:   {len(signoff_fields)}")
print()

if not errors:
    print("AUDIT RESULT: PASS — checklist is structurally complete.")
else:
    print(f"AUDIT RESULT: FAIL — {len(errors)} error(s) must be resolved.")

STRUCTURAL AUDIT — artifacts/evaluation_checklist.md

ERRORS: none

WARNINGS: none

Items audited:     39
Dimensions checked: 8
Sign-off fields:   7

AUDIT RESULT: PASS — checklist is structurally complete.


In [5]:
# Cell 5 — Human-readable summary and notebook completion record

from datetime import datetime

print("=" * 70)
print("EVALUATION CHECKLIST — ITEM SUMMARY")
print("=" * 70)
print()

current_dim = None
for item in checklist["dimensions"]:
    if item["dimension"] != current_dim:
        current_dim = item["dimension"]
        print(f"  {current_dim}")

    print(f"    {item['id']:<10}  {item['criterion']}")

print()
print("=" * 70)
print(f"Total items: {len(checklist['dimensions'])}")
print()

# ── Completion record ─────────────────────────────────────────────────────────
completion = {
    "notebook": "eda_12_evaluation_framework.ipynb",
    "completed": str(datetime.now().isoformat(timespec="seconds")),
    "output": "artifacts/evaluation_checklist.md",
    "items": len(checklist["dimensions"]),
    "dimensions": 8,
    "next_notebook": "eda_13_eda_finalization.ipynb",
    "next_goal": (
        "Consolidate all verdict CSVs into master_verdict.csv. "
        "Produce final_features.csv. Resolve all ambiguities. "
        "Write prior specification draft."
    ),
    "sign_off_notebook": "model_14_signoff.ipynb",
    "sign_off_day": 33,
}

completion_path = artifacts_dir / "eda_12_completion.json"
with open(completion_path, "w") as f:
    json.dump(completion, f, indent=2)

print(f"Completion record written: {completion_path}")
print()
print("Outputs produced:")
print(f"  artifacts/evaluation_checklist.md  — 39-item pass/fail checklist")
print(f"  artifacts/eda_12_completion.json   — completion record")
print()
print("Day 18 complete.")
print()
print("Next: Day 19 — eda_13_eda_finalization.ipynb")
print(f"  {completion['next_goal']}")

EVALUATION CHECKLIST — ITEM SUMMARY

  Convergence and Sampling Quality
    CONV-01     R-hat for all parameters
    CONV-02     Effective sample size
    CONV-03     Divergences
    CONV-04     Trace plots
    CONV-05     Energy plots (BFMI)
  Prior Predictive Checks
    PRIOR-01    Score range plausibility
    PRIOR-02    Score variance consistency with observed VMR
    PRIOR-03    Plausible total points distribution
  Posterior Predictive Checks
    POST-01     Overall score distribution fit
    POST-02     Conference-level calibration
    POST-03     Dispersion parameter adequacy
    POST-04     Tail behavior — high-scoring and low-scoring games
  Holdout Evaluation (2025 Season)
    HOLD-01     Overall Brier score vs baseline
    HOLD-02     Calibration curve — win probability
    HOLD-03     Spread accuracy by expected margin bucket
    HOLD-04     Over/under calibration
  Subgroup Evaluation
    SUB-01      Brier score by conference tier — P4
    SUB-02      Brier score by confe